# CUDA Acceleration Primitives

Portfolio notebook for validating CUDA C++ acceleration primitives used in AI/MLE systems work: matrix multiplication, shared-memory tiling, cuBLAS benchmarking, and Python-callable CUDA convolution.


## 1. Runtime Check

Confirm that Colab is using a GPU runtime with CUDA available.


In [2]:
!nvidia-smi
!nvcc --version


Mon Jun  1 05:36:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from pathlib import Path
import os
import subprocess
from google.colab import files

REPO_URL = "https://github.com/BrianChen29/High-Performance-CUDA-Programming.git"
REPO_DIR = Path("/content/High-Performance-CUDA-Programming")

if Path("collect_results.py").exists() and Path("convolution").exists():
    repo_root = Path.cwd()
else:
    repo_root = REPO_DIR
    if not repo_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)

os.chdir(repo_root)
print(f"Repo root: {Path.cwd()}")


Repo root: /content/High-Performance-CUDA-Programming


## 2. Matrix Multiplication Benchmarks

Compile and compare a CPU baseline, a naive CUDA kernel, a shared-memory tiled CUDA kernel, and a cuBLAS SGEMM reference.


In [4]:
!mkdir -p build
!gcc cpu/matrix_cpu.c -O2 -o build/matrix_cpu
!./build/matrix_cpu 1024


CPU execution time (N=1024): 3.463155 seconds


### Naive CUDA Kernel


In [5]:
!mkdir -p build
!nvcc -arch=sm_75 matrix_gpu.cu -o build/matrix_gpu
!./build/matrix_gpu 1024
!./build/matrix_gpu 2048
!./build/matrix_gpu 4096


Matrix Size N = 1024
GPU execution time (Naïve): 0.006869 seconds
Matrix Size N = 2048
GPU execution time (Naïve): 0.042030 seconds
Matrix Size N = 4096
GPU execution time (Naïve): 0.256699 seconds


### Shared-Memory Tiled CUDA Kernel


In [6]:
!mkdir -p build
!nvcc -arch=sm_75 matrix_gpu_optimized.cu -o build/matrix_gpu_optimized
!./build/matrix_gpu_optimized 1024
!./build/matrix_gpu_optimized 2048
!./build/matrix_gpu_optimized 4096


Matrix Size N = 1024 (Optimized Tiled CUDA)
GPU execution time (Tiled Optimization): 0.002213 seconds
Matrix Size N = 2048 (Optimized Tiled CUDA)
GPU execution time (Tiled Optimization): 0.017661 seconds
Matrix Size N = 4096 (Optimized Tiled CUDA)
GPU execution time (Tiled Optimization): 0.196805 seconds


### cuBLAS Reference


In [7]:
!mkdir -p build
!nvcc -arch=sm_75 matrix_cublas.cu -lcublas -o build/matrix_cublas
!./build/matrix_cublas 1024
!./build/matrix_cublas 2048
!./build/matrix_cublas 4096


Matrix Size N = 1024 (cuBLAS)
GPU execution time (cuBLAS): 0.063550 seconds
Matrix Size N = 2048 (cuBLAS)
GPU execution time (cuBLAS): 0.009544 seconds
Matrix Size N = 4096 (cuBLAS)
GPU execution time (cuBLAS): 0.033333 seconds


## 3. Automated Benchmark and Plots

Run the reproducible benchmark script and regenerate the matrix performance plots used in the README.


In [8]:
!python collect_results.py


Compiling all sources for CUDA architecture sm_75...
   Compiling CPU (cpu/matrix_cpu.c)...
   Compiling Naive GPU (matrix_gpu.cu)...
   Compiling Optimized (matrix_gpu_optimized.cu)...
   Compiling cuBLAS (matrix_cublas.cu)...
Compilation process finished.

Running tests for CPU...
   N=256: 0.021160 sec
   N=512: 0.314571 sec
   N=1024: 3.503723 sec
   N=2048: 75.699278 sec
   N=4096: Execution Timeout
------------------------------
Running tests for Naive GPU...
   N=256: 0.000264 sec
   N=512: 0.000592 sec
   N=1024: 0.004152 sec
   N=2048: 0.032821 sec
   N=4096: 0.262257 sec
------------------------------
Running tests for Optimized...
   N=256: 0.000146 sec
   N=512: 0.000342 sec
   N=1024: 0.002369 sec
   N=2048: 0.018067 sec
   N=4096: 0.194119 sec
------------------------------
Running tests for cuBLAS...
   N=256: 0.052196 sec
   N=512: 0.006267 sec
   N=1024: 0.006413 sec
   N=2048: 0.008388 sec
   N=4096: 0.037842 sec
------------------------------

All done! Results saved

In [9]:
files.download('results/final_comparison.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
!python plot_matrix_performance.py


Saved /content/High-Performance-CUDA-Programming/results/plot_log_scale.png (best for overall comparison)
Saved /content/High-Performance-CUDA-Programming/results/plot_gpu_only.png (best for analyzing CUDA optimizations)


In [11]:
files.download('results/plot_log_scale.png')
files.download('results/plot_gpu_only.png')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Python-Callable CUDA Matrix Multiplication

Compile the tiled matrix multiplication kernel as a shared library and call it from Python through `ctypes`.


In [12]:
!nvcc -arch=sm_75 -Xcompiler -fPIC -shared matrix_lib.cu -o libmatrix.so
!ls -lh libmatrix.so


-rwxr-xr-x 1 root root 981K Jun  1 05:40 libmatrix.so


In [13]:
files.download('libmatrix.so')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### ctypes Smoke Test


In [14]:
!python test_lib.py


Testing N=1024 with Python + CUDA...
-> Done in 0.3038 seconds

Testing N=2048 with Python + CUDA...
-> Done in 0.0499 seconds

Testing N=4096 with Python + CUDA...
-> Done in 0.2833 seconds



## 5. Python-Callable CUDA Convolution

Compile the convolution kernel as a shared library, then run the Python benchmark/visualization pipeline for Sobel edge detection and larger stencil filters.


In [15]:
!nvcc -arch=sm_75 -Xcompiler -fPIC -shared convolution/convolution_lib.cu -o convolution/libconvolution.so
!ls -lh convolution/libconvolution.so


-rwxr-xr-x 1 root root 981K Jun  1 05:40 convolution/libconvolution.so


In [16]:
files.download('convolution/libconvolution.so')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
!python convolution/test_convolution.py


Image download failed (HTTP Error 400: Use thumbnail sizes listed on https://w.wiki/GHai). Using random noise.
=== STARTING MULTI-SIZE PERFORMANCE TEST ===
Testing Image: 512x512 | Filter: Sobel X 3x3...
Testing Image: 1536x1536 | Filter: Sobel X 3x3...
Testing Image: 2560x2560 | Filter: Sobel X 3x3...
Testing Image: 1536x1536 | Filter: Box Blur 5x5...
Testing Image: 1536x1536 | Filter: Box Blur 7x7...

Experiment Type      | Image Size      | Filter   | GPU (s)    | CPU (s)    | Speedup 
--------------------------------------------------------------------------------
Varying Image Size   | 512x512         | Sobel 3x3 | 0.0012     | 0.0148     | 12.1x
Varying Image Size   | 1536x1536       | Sobel 3x3 | 0.0056     | 0.1461     | 25.9x
Varying Image Size   | 2560x2560       | Sobel 3x3 | 0.0146     | 0.3671     | 25.1x
Varying Filter Size  | 1536x1536       | Box 5x5  | 0.0057     | 0.3495     | 60.9x
Varying Filter Size  | 1536x1536       | Box 7x7  | 0.0062     | 0.6814     | 109.4x



In [18]:
files.download('convolution/results/edge_detection_result.png')
files.download('convolution/results/convolution_output_demo.png')
files.download('convolution/results/comparison.png')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Standalone CUDA Convolution Benchmark

Run the pure CUDA executable as a reference point for Python wrapper overhead.


In [19]:
!nvcc -arch=sm_75 convolution/convolution_standalone.cu -o convolution/convolution_standalone
!./convolution/convolution_standalone


Benchmarking Standalone CUDA...
Image Size: 4000 x 2665 (Matching Python Mosaic)
------------------------------------------------
Direct CUDA Executable Time: 0.0011 seconds
------------------------------------------------
